In [ ]:
import os
import csv
import pandas as pd

# ---------- Config ----------
dataset_folder = r"C:\git\energy-forcaster\datasets"
datasets = [
    "wb_gdp_data.csv",
    "wb_renewable_energy.csv",
    "wb_population_data.csv",
    "wb_energy_use.csv",
    "kaggle_energy_data.csv",
    "kaggle_weather.csv",
]

# ---------- Helpers ----------
def sniff_delimiter(sample_bytes: bytes, default=','):
    """Try to guess the delimiter from a byte sample."""
    try:
        dialect = csv.Sniffer().sniff(sample_bytes.decode('utf-8', errors='ignore'), delimiters=[',',';','\t','|'])
        return dialect.delimiter
    except Exception:
        return default

def robust_read_csv(path: str) -> pd.DataFrame:
    """
    Read a CSV with several fallbacks:
    1) Default fast engine
    2) Auto-detected delimiter (python engine)
    3) UTF-8-SIG / Latin-1 encodings
    4) Skip bad lines if necessary
    """
    # 1) Try plain read
    try:
        return pd.read_csv(path, low_memory=False)
    except Exception as e1:
        last_err = e1

    # Prepare a small sample to sniff delimiter
    delim = ','
    try:
        with open(path, 'rb') as f:
            sample = f.read(65536)
        delim = sniff_delimiter(sample, default=',')
    except Exception:
        pass

    # Candidates to try
    attempts = [
        dict(sep=delim, engine='python', encoding='utf-8', on_bad_lines='error'),
        dict(sep=delim, engine='python', encoding='utf-8-sig', on_bad_lines='error'),
        dict(sep=delim, engine='python', encoding='latin-1', on_bad_lines='error'),
        # As a last resort, skip malformed lines
        dict(sep=delim, engine='python', encoding='utf-8', on_bad_lines='skip'),
        dict(sep=delim, engine='python', encoding='utf-8-sig', on_bad_lines='skip'),
        dict(sep=delim, engine='python', encoding='latin-1', on_bad_lines='skip'),
    ]

    for i, opts in enumerate(attempts, 1):
        try:
            df = pd.read_csv(path, low_memory=False, **opts)
            print(f"[info] Loaded with options #{i}: {opts}")
            return df
        except Exception as ex:
            last_err = ex

    # If all failed, re-raise the last error for visibility
    raise last_err

def coerce_numeric_like(df: pd.DataFrame) -> pd.DataFrame:
    """
    Convert object columns that look numeric into numeric, safely.
    Uses coercion but reverts if it destroys too much data.
    """
    for col in df.columns:
        if df[col].dtype == 'object':
            non_null = df[col].notna().sum()
            # Try coercion
            coerced = pd.to_numeric(df[col].str.replace(',', ''), errors='coerce')
            # Accept if it produced *some* numbers and didn't wipe out most data
            if (coerced.notna().sum() >= max(5, 0.5 * non_null)) or (coerced.notna().sum() >= 0.9 * non_null):
                df[col] = coerced
    return df

def clean_data(file_name: str):
    file_path = os.path.join(dataset_folder, file_name)
    if not os.path.exists(file_path):
        print(f"[warn] File not found: {file_path}")
        return

    # Read (robust)
    df = robust_read_csv(file_path)

    # 1) Remove duplicate rows
    before = len(df)
    df = df.drop_duplicates()
    after = len(df)
    if before != after:
        print(f"[info] Dropped {before - after} duplicate rows from {file_name}")

    # 2) Missing values: keep NaN as-is (structure preserved)

    # 3) Convert numeric-looking columns
    df = coerce_numeric_like(df)

    # 4) Save cleaned dataset
    cleaned_file_path = os.path.join(dataset_folder, f"cleaned_{file_name}")
    df.to_csv(cleaned_file_path, index=False)
    print(f"[ok] Cleaned file saved: {cleaned_file_path} | shape={df.shape}")

# ---------- Run ----------
for f in datasets:
    try:
        clean_data(f)
    except Exception as e:
        print(f"[error] Failed cleaning {f}: {e}")

print("All datasets processed.")



In [ ]:
import os
import pandas as pd

# ----------------- Config -----------------
dataset_folder = r"C:\git\energy-forcaster\datasets"  # your path
datasets = [
    "cleaned_wb_gdp_data.csv",
    "cleaned_wb_renewable_energy.csv",
    "cleaned_wb_population_data.csv",
    "cleaned_wb_energy_use.csv",
    "cleaned_kaggle_energy_data.csv",
    "cleaned_kaggle_weather.csv",
]

# ----------------- Helpers -----------------
POSSIBLE_COUNTRY = {"country", "country_name", "nation"}
POSSIBLE_YEAR    = {"year", "yr", "fiscal_year"}

def standardize_columns(df: pd.DataFrame) -> pd.DataFrame:
    # lowercase + replace whitespace with underscore
    df = df.rename(columns=lambda c: c.strip().lower().replace(" ", "_"))
    return df

def rename_keys(df: pd.DataFrame) -> pd.DataFrame:
    cols = set(df.columns)

    # Find and rename country column
    country_col = next((c for c in POSSIBLE_COUNTRY if c in cols), None)
    if country_col and country_col != "country":
        df = df.rename(columns={country_col: "country"})

    # Find and rename year column
    year_col = next((c for c in POSSIBLE_YEAR if c in cols), None)
    if year_col and year_col != "year":
        df = df.rename(columns={year_col: "year"})

    return df

def read_and_prepare(file_name: str):
    file_path = os.path.join(dataset_folder, file_name)
    if not os.path.exists(file_path):
        print(f"[warn] File not found, skipping: {file_path}")
        return None

    try:
        df = pd.read_csv(file_path, low_memory=False)
    except Exception as e:
        print(f"[warn] Could not read {file_name}: {e}")
        return None

    df = standardize_columns(df)
    df = rename_keys(df)

    # Require 'country' and 'year'
    if "country" not in df.columns or "year" not in df.columns:
        print(f"[info] Skipping {file_name} - missing 'country' or 'year' column")
        return None

    return df

# ----------------- Load & Merge -----------------
dataframes = []
for f in datasets:
    df = read_and_prepare(f)
    if df is not None:
        dataframes.append(df)

if dataframes:
    # Full outer join on country & year across all frames
    from functools import reduce
    merged = reduce(lambda left, right: pd.merge(left, right, on=["country", "year"], how="outer"), dataframes)

    # Missing values are already NaN in pandas; no special action needed
    out_path = os.path.join(dataset_folder, "merged_energy_data.csv")
    merged.to_csv(out_path, index=False)
    print(f"[ok] Merged dataset saved at: {out_path} | shape={merged.shape}")
else:
    print("[info] No datasets could be merged due to missing 'country' or 'year'.")


In [ ]:
# === Setup ===
import os
import numpy as np
import pandas as pd
import plotly.graph_objects as go
from ipywidgets import Dropdown, VBox, HBox, HTML, Output
from IPython.display import display, clear_output

# ---- Paths ----
dataset_folder = r"C:\git\energy-forcaster\datasets"
merged_data_path = os.path.join(dataset_folder, "merged_energy_data.csv")

# ---- Load data ----
df = pd.read_csv(merged_data_path, low_memory=False)

# normalize column names a bit (optional but helpful)
df.columns = [c.strip() for c in df.columns]
if 'year' in df.columns:
    # Ensure year is numeric for correct sorting/plotting
    df['year'] = pd.to_numeric(df['year'], errors='coerce')

# ---- Available fields (filter by what exists) ----
energy_types_all = [
    "primary_energy_consumption",
    "fossil_fuel_consumption",
    "renewables_consumption",
    "oil_consumption",
    "nuclear_consumption",
]
energy_types = [c for c in energy_types_all if c in df.columns]

corr_candidates_all = [
    "gdp",
    "population",
    "fossil_fuel_consumption",
    "renewables_consumption",
    "energy_per_gdp",
    "average_temperature",
]
correlation_factors = [c for c in corr_candidates_all if c in df.columns]

# ---- Country list ----
if 'country' not in df.columns:
    raise ValueError("The merged dataset must contain a 'country' column.")
country_list = sorted([c for c in df['country'].dropna().unique().tolist()])

# ---- Widgets ----
default_country = "United States" if "United States" in country_list else (country_list[0] if country_list else None)
country_dd = Dropdown(
    options=country_list,
    value=default_country,
    description="Primary Country:",
    layout={'width': '400px'}
)

compare_dd = Dropdown(
    options=["None"] + country_list,
    value="None",
    description="Compare With:",
    layout={'width': '400px'}
)

energy_dd = Dropdown(
    options=energy_types,
    value=energy_types[0] if energy_types else None,
    description="Energy Type:",
    layout={'width': '400px'}
)

var_dd = Dropdown(
    options=correlation_factors,
    value=correlation_factors[0] if correlation_factors else None,
    description="Factor:",
    layout={'width': '400px'}
)

corr_html = HTML(value="<b>Correlation:</b> —")
trend_out = Output()
scatter_out = Output()

# ---- Helpers ----
def line_fit(x, y):
    """Return x_line, y_line for simple linear regression line using numpy polyfit."""
    mask = (~pd.isna(x)) & (~pd.isna(y))
    xv, yv = x[mask], y[mask]
    if len(xv) < 2:
        return None, None
    m, b = np.polyfit(xv, yv, 1)
    xs = np.linspace(xv.min(), xv.max(), 100)
    ys = m * xs + b
    return xs, ys

def interpret_corr(r):
    if r is None or np.isnan(r):
        return "Not enough data to compute correlation."
    if r > 0.7:
        return "Strong positive correlation - The factor has a significant positive impact on energy consumption."
    if r < -0.7:
        return "Strong negative correlation - The factor significantly reduces energy consumption."
    if abs(r) > 0.4:
        return "Moderate correlation - The factor somewhat affects energy consumption."
    return "Weak or no correlation - This factor does not strongly impact energy consumption."

# ---- Update functions ----
def update_trend(*_):
    with trend_out:
        clear_output(wait=True)
        energy_col = energy_dd.value
        if energy_col is None or 'year' not in df.columns:
            display(HTML("<i>No energy column or 'year' available to plot.</i>"))
            return

        traces = []
        title_bits = [f"Energy Consumption Trends: {energy_col}"]
        # Primary
        if country_dd.value:
            cdat = df[(df['country'] == country_dd.value) & (~df[energy_col].isna())].copy()
            cdat = cdat.sort_values('year')
            if not cdat.empty:
                traces.append(go.Scatter(
                    x=cdat['year'], y=cdat[energy_col],
                    mode='lines+markers', name=country_dd.value
                ))
        # Compare
        if compare_dd.value and compare_dd.value != "None":
            c2 = df[(df['country'] == compare_dd.value) & (~df[energy_col].isna())].copy()
            c2 = c2.sort_values('year')
            if not c2.empty:
                traces.append(go.Scatter(
                    x=c2['year'], y=c2[energy_col],
                    mode='lines+markers', name=compare_dd.value
                ))
            title_bits.append(f"(vs {compare_dd.value})")

        fig = go.Figure(data=traces)
        fig.update_layout(
            title=" ".join(title_bits),
            xaxis_title="Year",
            yaxis_title=energy_col,
            template="plotly_white",
            legend_title="Country"
        )
        fig.show()

def update_scatter_and_corr(*_):
    with scatter_out:
        clear_output(wait=True)
        energy_col = "primary_energy_consumption"
        xcol = var_dd.value

        if energy_col not in df.columns:
            display(HTML("<i>'primary_energy_consumption' column not found in data.</i>"))
            corr_html.value = "<b>Correlation:</b> —"
            return
        if xcol is None or xcol not in df.columns:
            display(HTML("<i>Selected factor is not available in the dataset.</i>"))
            corr_html.value = "<b>Correlation:</b> —"
            return

        # Filter rows with both values
        work = df[[xcol, energy_col]].dropna()
        if work.empty or work[xcol].nunique() < 2 or work[energy_col].nunique() < 2:
            display(HTML("<i>Not enough data to plot scatter or compute correlation.</i>"))
            corr_html.value = "<b>Correlation:</b> —"
            return

        # Pearson r
        r = work[xcol].corr(work[energy_col])
        interp = interpret_corr(r)
        corr_html.value = f"<b>Correlation:</b> {r:.3f} — {interp}"

        # Scatter + regression
        xs_line, ys_line = line_fit(work[xcol], work[energy_col])

        fig = go.Figure()
        fig.add_trace(go.Scatter(
            x=work[xcol], y=work[energy_col],
            mode='markers', name='Data', opacity=0.6
        ))
        if xs_line is not None:
            fig.add_trace(go.Scatter(
                x=xs_line, y=ys_line,
                mode='lines', name='Linear fit'
            ))

        fig.update_layout(
            title=f"Energy Consumption vs {xcol}",
            xaxis_title=xcol,
            yaxis_title="Primary Energy Consumption (TWh)",
            template="plotly_white"
        )
        fig.show()

# ---- Wire up events ----
for w in (country_dd, compare_dd, energy_dd):
    w.observe(update_trend, names='value')

var_dd.observe(update_scatter_and_corr, names='value')

# ---- Initial render ----
update_trend()
update_scatter_and_corr()

# ---- Layout ----
ui = VBox([
    HTML("<h2>📊 Interactive Energy Consumption Analysis</h2>"),
    HTML("<h3>🔹 Task 1.1: Energy Consumption Trends</h3>"),
    HBox([country_dd, compare_dd, energy_dd]),
    trend_out,
    HTML("<hr>"),
    HTML("<h3>🔹 Task 1.2: Correlation Analysis</h3>"),
    HBox([var_dd, corr_html]),
    scatter_out,
    HTML("<hr>"),
    HTML("<h3>📘 How Correlation is Calculated</h3>"
         "<pre>r = Σ [(Xi - X̄)(Yi - Ȳ)] / sqrt(Σ (Xi - X̄)² * Σ (Yi - Ȳ)²)</pre>"
         "<ul>"
         "<li><b>+1</b> → Perfect positive correlation</li>"
         "<li><b>0</b> → No correlation</li>"
         "<li><b>-1</b> → Perfect negative correlation</li>"
         "</ul>")
])

display(ui)


In [ ]:
# === Feature Engineering for Energy Dataset (Python / pandas) ===
import os
import numpy as np
import pandas as pd

# ---------------- Config ----------------
dataset_folder = r"C:\git\energy-forcaster\datasets"
merged_data_path = os.path.join(dataset_folder, "merged_energy_data.csv")
out_path = os.path.join(dataset_folder, "feature_engineered_energy_data.csv")

# ---------------- Load ----------------
df = pd.read_csv(merged_data_path, low_memory=False)

# Ensure 'year' numeric and sort by country/year (stable sort to preserve order ties)
df['year'] = pd.to_numeric(df.get('year', np.nan), errors='coerce')
df = df.sort_values(['country', 'year'], kind='mergesort').reset_index(drop=True)

# Convenience flags for columns that may or may not exist
has_primary = 'primary_energy_consumption' in df.columns
has_temp    = 'average_temperature' in df.columns
has_gdp     = 'gdp' in df.columns
has_pop     = 'population' in df.columns
has_ren     = 'renewables_consumption' in df.columns

# ---------------- Derived Features ----------------

# 1) Lagged Energy (1-year) within each country
df['lagged_energy'] = (
    df.groupby('country')['primary_energy_consumption'].shift(1)
    if has_primary else np.nan
)

# 2) Moving Average (3-year, right aligned) within each country
if has_primary:
    df['moving_avg_energy'] = (
        df.groupby('country')['primary_energy_consumption']
          .rolling(window=3, min_periods=3)
          .mean()
          .reset_index(level=0, drop=True)
    )
else:
    df['moving_avg_energy'] = np.nan

# 3) Year-over-Year Growth (%) within each country
if has_primary:
    g = df.groupby('country')['primary_energy_consumption']
    df['yoy_growth'] = (g.transform(lambda s: s.diff(1)) / g.shift(1))
else:
    df['yoy_growth'] = np.nan

# 4) Decade indicator (e.g., 1990, 2000, …). Keep as float if year is missing.
decade = np.floor(df['year'] / 10) * 10
df['decade'] = decade

# 5) Seasonal Indicators
#    Your R code builds a date as Jan 1 of each year; month will be 1 for all rows with valid year.
#    We keep the same logic explicitly (month=1 when year is valid), then flags.
month_series = pd.to_datetime(
    df['year'].dropna().astype(int).astype(str) + "-01-01",
    errors='coerce'
).dt.month
df['month'] = np.nan
df.loc[month_series.index, 'month'] = month_series
# Convert to simple integers where possible
df['month'] = df['month'].fillna(1).astype(int)  # default to January if year missing
df['is_winter'] = np.where(df['month'].isin([12, 1, 2]), 1, 0).astype(int)
df['is_summer'] = np.where(df['month'].isin([6, 7, 8]), 1, 0).astype(int)

# 6) Weather-Based Features (optional)
if has_temp:
    avg = pd.to_numeric(df['average_temperature'], errors='coerce')
    df['temp_anomaly'] = avg - avg.mean(skipna=True)
    df['HDD'] = np.where(avg < 18, 18 - avg, 0)
    df['CDD'] = np.where(avg > 24, avg - 24, 0)
else:
    df['temp_anomaly'] = np.nan
    df['HDD'] = np.nan
    df['CDD'] = np.nan

# 7) Energy Efficiency Features (safe division)
def safe_ratio(numer: pd.Series, denom: pd.Series) -> pd.Series:
    return pd.to_numeric(numer, errors='coerce') / pd.to_numeric(denom, errors='coerce')

if has_primary and has_gdp:
    df['energy_intensity'] = safe_ratio(df['primary_energy_consumption'], df['gdp'])
else:
    df['energy_intensity'] = np.nan

if has_primary and has_pop:
    df['energy_per_capita'] = safe_ratio(df['primary_energy_consumption'], df['population'])
else:
    df['energy_per_capita'] = np.nan

if has_ren and has_primary:
    df['renewable_share'] = safe_ratio(df['renewables_consumption'], df['primary_energy_consumption'])
else:
    df['renewable_share'] = np.nan

# 8) Rolling Aggregations (5-year, right aligned) within each country
if has_primary:
    df['rolling_avg_5yr'] = (
        df.groupby('country')['primary_energy_consumption']
          .rolling(window=5, min_periods=5)
          .mean()
          .reset_index(level=0, drop=True)
    )
    df['rolling_std_5yr'] = (
        df.groupby('country')['primary_energy_consumption']
          .rolling(window=5, min_periods=5)
          .std()
          .reset_index(level=0, drop=True)
    )
else:
    df['rolling_avg_5yr'] = np.nan
    df['rolling_std_5yr'] = np.nan

# ---------------- Scaling & Normalization ----------------

def min_max_scale(x: pd.Series) -> pd.Series:
    """Min-Max scaling with NA/constant handling."""
    x = pd.to_numeric(x, errors='coerce')
    if x.isna().all():
        return x
    minv = x.min(skipna=True)
    maxv = x.max(skipna=True)
    rng = maxv - minv
    if pd.isna(minv) or pd.isna(maxv) or rng == 0:
        return x
    return (x - minv) / rng

def z_score_norm(x: pd.Series) -> pd.Series:
    """Z-score normalization with NA/zero-std handling."""
    x = pd.to_numeric(x, errors='coerce')
    if x.isna().all():
        return x
    mean = x.mean(skipna=True)
    std = x.std(skipna=True)
    if pd.isna(mean) or pd.isna(std) or std == 0:
        return x
    return (x - mean) / std

numeric_columns = [
    "primary_energy_consumption", "gdp", "population", "fossil_fuel_consumption",
    "renewables_consumption", "energy_per_gdp", "lagged_energy", "moving_avg_energy",
    "yoy_growth", "energy_intensity", "energy_per_capita", "renewable_share",
    "rolling_avg_5yr", "rolling_std_5yr", "average_temperature", "temp_anomaly", "HDD", "CDD"
]
numeric_columns = [c for c in numeric_columns if c in df.columns]

for col in numeric_columns:
    df[f"scaled_{col}"] = min_max_scale(df[col])
    df[f"normalized_{col}"] = z_score_norm(df[col])

# ---------------- Save ----------------
df.to_csv(out_path, index=False)
print(f"Feature Engineering Completed! File saved at: {out_path}")



In [14]:
pip install pandas plotly ipywidgets statsmodels scipy scikit-learn


In [12]:
# === All-Country Model Comparison with Verbose Logging ===
# Models: ARIMA, Exponential Smoothing, Random Forest, XGBoost (or GradientBoosting fallback)

import os
import numpy as np
import pandas as pd
from collections import Counter
import warnings

from sklearn.metrics import mean_squared_error
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor

# Optional: in-notebook install (uncomment if needed)
# %pip install xgboost tqdm

# Try to import xgboost; if unavailable, use GradientBoostingRegressor as a drop-in
try:
    from xgboost import XGBRegressor
    HAS_XGB = True
except Exception:
    XGBRegressor = GradientBoostingRegressor
    HAS_XGB = False

# Progress bar (optional)
try:
    from tqdm.auto import tqdm
    TQDM = True
except Exception:
    TQDM = False

from statsmodels.tsa.arima.model import ARIMA
from statsmodels.tsa.holtwinters import ExponentialSmoothing

warnings.filterwarnings("ignore")

# ---------------- Paths ----------------
dataset_folder = r"C:\git\energy-forcaster\datasets"
data_path = os.path.join(dataset_folder, "feature_engineered_energy_data.csv")
out_csv_path = os.path.join(dataset_folder, "model_results_by_country.csv")
best_csv_path = os.path.join(dataset_folder, "best_model_per_country.csv")

# ---------------- Load Data ----------------
print(f"[INFO] Loading dataset: {data_path}")
if not os.path.exists(data_path):
    raise FileNotFoundError(f"File not found: {data_path}")

df = pd.read_csv(data_path, low_memory=False)
required_cols = {"country", "year", "primary_energy_consumption"}
missing_req = required_cols - set(df.columns)
if missing_req:
    raise ValueError(f"Missing required columns: {missing_req}")

df["year"] = pd.to_numeric(df["year"], errors="coerce")
df = df.sort_values(["country", "year"]).reset_index(drop=True)

countries_all = df["country"].dropna().unique().tolist()
print(f"[INFO] Countries found: {len(countries_all)}")

# Quick diagnostics
counts = (
    df.groupby("country")["primary_energy_consumption"]
      .apply(lambda s: pd.to_numeric(s, errors="coerce").dropna().shape[0])
      .reset_index(name="non_null_points")
      .sort_values("non_null_points")
)
too_short = counts[counts["non_null_points"] < 12]
if not too_short.empty:
    print("[INFO] Countries with < 12 points (will be skipped):")
    print(too_short.to_string(index=False))

target_col = "primary_energy_consumption"

# ---------------- Helpers ----------------
def safe_mape(y_true, y_pred, eps=1e-8):
    y_true = np.asarray(y_true, dtype=float)
    y_pred = np.asarray(y_pred, dtype=float)
    denom = np.maximum(np.abs(y_true), eps)
    return np.mean(np.abs((y_true - y_pred) / denom)) * 100.0

def evaluate(y_true, y_pred):
    rmse = float(np.sqrt(mean_squared_error(y_true, y_pred)))
    mape = float(safe_mape(y_true, y_pred))
    return rmse, mape

def make_lag_supervised(series: pd.Series, n_lags=3):
    frame = pd.DataFrame({f"lag_{i}": series.shift(i) for i in range(1, n_lags+1)})
    frame[target_col] = series
    frame = frame.dropna()
    X = frame.drop(columns=[target_col])
    y = frame[target_col]
    return X, y

def time_split(y_series: pd.Series, test_frac=0.2, min_test=6):
    n = len(y_series)
    t = max(min_test, int(np.ceil(test_frac * n)))
    if n <= t:
        return None, None
    return y_series.iloc[:-t], y_series.iloc[-t:]

def fit_predict_arima(train, test, order=(2,1,2)):
    fit = ARIMA(train, order=order).fit()
    pred = fit.forecast(steps=len(test))
    return np.array(pred)

def fit_predict_ets(train, test):
    fit = ExponentialSmoothing(train, trend='add', seasonal=None).fit()
    pred = fit.forecast(steps=len(test))
    return np.array(pred)

def fit_predict_rf(y, train_len, n_lags=3):
    X_all, y_all = make_lag_supervised(y, n_lags=n_lags)
    n_shift = len(y) - len(y_all)
    split_idx = train_len - n_shift
    if split_idx <= 0 or split_idx >= len(y_all):
        raise ValueError("Insufficient data after lagging for RF split.")
    X_tr, X_te = X_all.iloc[:split_idx], X_all.iloc[split_idx:]
    y_tr, y_te = y_all.iloc[:split_idx], y_all.iloc[split_idx:]
    rf = RandomForestRegressor(n_estimators=300, random_state=42, n_jobs=-1)
    rf.fit(X_tr, y_tr)
    pred = rf.predict(X_te)
    return y_te.values, pred

def fit_predict_xgb(y, train_len, n_lags=3):
    X_all, y_all = make_lag_supervised(y, n_lags=n_lags)
    n_shift = len(y) - len(y_all)
    split_idx = train_len - n_shift
    if split_idx <= 0 or split_idx >= len(y_all):
        raise ValueError("Insufficient data after lagging for XGB split.")
    X_tr, X_te = X_all.iloc[:split_idx], X_all.iloc[split_idx:]
    y_tr, y_te = y_all.iloc[:split_idx], y_all.iloc[split_idx:]
    if HAS_XGB:
        xgb = XGBRegressor(
            n_estimators=500, learning_rate=0.05, max_depth=3,
            subsample=0.8, colsample_bytree=0.8, random_state=42
        )
        label = "XGBoost"
    else:
        xgb = XGBRegressor(  # GradientBoostingRegressor alias
            n_estimators=500, learning_rate=0.05, max_depth=3, random_state=42
        )
        label = "Gradient Boosting (XGB substitute)"
    xgb.fit(X_tr, y_tr)
    pred = xgb.predict(X_te)
    return y_te.values, pred, label

# ---------------- Per-Country Evaluation ----------------
rows = []
skipped = []

iterator = tqdm(countries_all, desc="Evaluating", leave=True) if TQDM else countries_all
for country in iterator:
    sub = df[df["country"] == country][["year", target_col]].copy()
    sub[target_col] = pd.to_numeric(sub[target_col], errors="coerce")
    sub = sub.dropna().sort_values("year")
    n_points = len(sub)

    if n_points < 12:
        skipped.append((country, f"{n_points} points (<12)"))
        continue

    y = sub[target_col].reset_index(drop=True)
    train, test = time_split(y, test_frac=0.2, min_test=6)
    if train is None:
        skipped.append((country, "time split failed"))
        continue

    # ARIMA
    try:
        pred = fit_predict_arima(train, test, order=(2,1,2))
        rmse, mape = evaluate(test, pred)
        rows.append({"country": country, "model": "ARIMA", "RMSE": rmse, "MAPE": mape})
    except Exception as e:
        rows.append({"country": country, "model": "ARIMA", "RMSE": np.nan, "MAPE": np.nan})

    # ETS
    try:
        pred = fit_predict_ets(train, test)
        rmse, mape = evaluate(test, pred)
        rows.append({"country": country, "model": "Exponential Smoothing", "RMSE": rmse, "MAPE": mape})
    except Exception as e:
        rows.append({"country": country, "model": "Exponential Smoothing", "RMSE": np.nan, "MAPE": np.nan})

    # RF
    try:
        y_true, y_pred = fit_predict_rf(y, train_len=len(train), n_lags=3)
        rmse, mape = evaluate(y_true, y_pred)
        rows.append({"country": country, "model": "Random Forest", "RMSE": rmse, "MAPE": mape})
    except Exception as e:
        rows.append({"country": country, "model": "Random Forest", "RMSE": np.nan, "MAPE": np.nan})

    # XGB / GB
    try:
        y_true, y_pred, label = fit_predict_xgb(y, train_len=len(train), n_lags=3)
        rmse, mape = evaluate(y_true, y_pred)
        rows.append({"country": country, "model": label, "RMSE": rmse, "MAPE": mape})
    except Exception as e:
        label = "XGBoost" if HAS_XGB else "Gradient Boosting (XGB substitute)"
        rows.append({"country": country, "model": label, "RMSE": np.nan, "MAPE": np.nan})

# ---------------- Aggregate Results ----------------
if not rows:
    print("\n[ERROR] No results produced. Possible reasons:")
    print(" - All countries had < 12 usable data points")
    print(" - Required columns missing or entirely NaN")
    print(" - Fatal errors during model fitting (check logs above)")
else:
    results_df = pd.DataFrame(rows)
    results_df = results_df.sort_values(["country", "model"])

    # Best model per country by RMSE (ignoring NaNs)
    best_per_country = (
        results_df
        .dropna(subset=["RMSE"])
        .sort_values(["country", "RMSE"])
        .groupby("country", as_index=False)
        .first()
        .rename(columns={"model": "best_model_by_RMSE"})
    )

    # Overall best by most wins; tie-breaker: lowest median RMSE
    if not best_per_country.empty:
        win_counts = Counter(best_per_country["best_model_by_RMSE"])
        max_wins = max(win_counts.values())
        contenders = [m for m, w in win_counts.items() if w == max_wins]

        if len(contenders) == 1:
            overall_best = contenders[0]
        else:
            med_rmse = (
                results_df
                .dropna(subset=["RMSE"])
                .groupby("model")["RMSE"]
                .median()
                .to_dict()
            )
            overall_best = min(contenders, key=lambda m: med_rmse.get(m, np.inf))
    else:
        overall_best = "N/A"

    # ---------------- Save & Print ----------------
    results_df.to_csv(out_csv_path, index=False)
    best_per_country.to_csv(best_csv_path, index=False)

    print("\n=== RMSE & MAPE by Model and Country (head) ===")
    print(results_df.head(20).to_string(index=False))

    print("\n=== Best Model per Country (by RMSE) ===")
    if best_per_country.empty:
        print("No valid results to summarize.")
    else:
        print(best_per_country.to_string(index=False))

    print("\n=== Overall Best Model ===")
    print(f"🏆 Overall best model for forecasting {target_col} across countries: {overall_best}")

    print(f"\n[OK] Full results saved to: {out_csv_path}")
    print(f"[OK] Best-per-country saved to: {best_csv_path}")

# ---------------- Skipped Countries Report ----------------
if skipped:
    print("\n[INFO] Skipped countries:")
    for c, reason in skipped:
        print(f" - {c}: {reason}")
else:
    print("\n[INFO] No countries were skipped.")




[INFO] Loading dataset: C:\git\energy-forcaster\datasets\feature_engineered_energy_data.csv
[INFO] Countries found: 294
[INFO] Countries with < 12 points (will be skipped):
                              country  non_null_points
                        ASEAN (Ember)                0
                         Asia (Ember)                0
             Asia and Oceania (Shift)                0
                 Persian Gulf (Shift)                0
                            OPEC (EI)                0
                         OPEC (Shift)                0
                  Middle East (Ember)                0
                  Middle East (Shift)                0
                        Non-OPEC (EI)                0
                North America (Ember)                0
                North America (Shift)                0
                      Oceania (Ember)                0
                         OECD (Ember)                0
                         OECD (Shift)                0
  

Evaluating:   0%|          | 0/294 [00:00<?, ?it/s]


=== RMSE & MAPE by Model and Country (head) ===
     country                              model       RMSE      MAPE
 Afghanistan                              ARIMA   8.437512 27.529384
 Afghanistan              Exponential Smoothing  30.947379 97.792903
 Afghanistan Gradient Boosting (XGB substitute)  14.755745 43.490472
 Afghanistan                      Random Forest   7.681815 20.586858
      Africa                              ARIMA 211.181607  3.550421
      Africa              Exponential Smoothing 186.042724  3.173616
      Africa Gradient Boosting (XGB substitute) 907.389915 15.082566
      Africa                      Random Forest 945.014947 15.861351
 Africa (EI)                              ARIMA 211.180896  3.550414
 Africa (EI)              Exponential Smoothing 186.154333  3.175464
 Africa (EI) Gradient Boosting (XGB substitute) 907.390018 15.082566
 Africa (EI)                      Random Forest 945.015023 15.861351
Africa (EIA)                              ARIMA 356.05

In [18]:
# === Exponential Smoothing Energy Forecast (Version-robust, no SciPy) ===
# Path: C:\git\energy-forcaster\datasets

import os
import numpy as np
import pandas as pd
import plotly.graph_objects as go
from ipywidgets import Dropdown, Button, VBox, HBox, Output, HTML
from IPython.display import display, clear_output
from sklearn.metrics import mean_squared_error, mean_absolute_percentage_error
from statsmodels.tsa.holtwinters import ExponentialSmoothing
import warnings
warnings.filterwarnings("ignore")

# ----------------- Config -----------------
dataset_folder = r"C:\git\energy-forcaster\datasets"
feature_path   = os.path.join(dataset_folder, "feature_engineered_energy_data.csv")

# ----------------- Load -------------------
if not os.path.exists(feature_path):
    raise FileNotFoundError(f"Feature file not found: {feature_path}")

df = pd.read_csv(feature_path, low_memory=False)
df.columns = [c.strip() for c in df.columns]

possible_energy_columns = [
    "energy_consumption",
    "primary_energy_consumption",
    "total_energy_consumption",
    "electricity_demand",
]
energy_candidates = [c for c in possible_energy_columns if c in df.columns]
if not energy_candidates:
    raise ValueError("No valid energy consumption column found in the dataset.")
energy_col = energy_candidates[0]
print(f"[INFO] Using column: {energy_col} for energy consumption")

if "country" not in df.columns or "year" not in df.columns:
    raise ValueError("The dataset must contain 'country' and 'year' columns.")

df["year"] = pd.to_numeric(df["year"], errors="coerce")
df = df.sort_values(["country", "year"]).reset_index(drop=True)
df = df[~df[energy_col].isna()]

countries = sorted(df["country"].dropna().unique())

# ----------------- Widgets ----------------
country_dd = Dropdown(options=countries, description="Country:", layout={"width": "350px"})
run_btn = Button(description="Run Forecast", button_style="primary")
metrics_out = Output()
plot_out = Output()
table_out = Output()
title_html = HTML("<h3>Exponential Smoothing Energy Consumption Forecast (Performance Metrics)</h3>")

# ----------------- Helpers ----------------
def rmse(y_true, y_pred):
    return float(np.sqrt(mean_squared_error(y_true, y_pred)))

def mape_pct(y_true, y_pred):
    return float(mean_absolute_percentage_error(np.asarray(y_true, float),
                                               np.asarray(y_pred, float)) * 100.0)

def interval_from_resid(point_fc: np.ndarray, resid_std: float, z: float):
    """Symmetric intervals: point ± z*std; safe if std==0."""
    if resid_std <= 0 or np.isnan(resid_std):
        lower = point_fc.copy()
        upper = point_fc.copy()
    else:
        lower = point_fc - z * resid_std
        upper = point_fc + z * resid_std
    return lower, upper

def fit_ets_robust(y_pos: pd.Series):
    """
    Try ETS with Box-Cox at model init (newer statsmodels).
    If that fails (older versions), retry WITHOUT Box-Cox.
    Always return a fitted model.
    """
    # 1) Try with Box-Cox at init (only if strictly positive)
    if np.nanmin(y_pos) > 0:
        try:
            model = ExponentialSmoothing(
                y_pos, trend="add", seasonal=None,
                initialization_method="estimated",
                use_boxcox="auto"  # let statsmodels choose lambda
            )
            return model.fit(optimized=True)
        except TypeError:
            # Older versions don't accept use_boxcox at init — fall through
            pass
        except Exception:
            # Any other failure → try without Box-Cox
            pass

    # 2) Fallback: no Box-Cox
    model = ExponentialSmoothing(
        y_pos, trend="add", seasonal=None,
        initialization_method="estimated"
    )
    return model.fit(optimized=True)

# ----------------- Action -----------------
def run_forecast(_):
    with metrics_out:
        clear_output(wait=True)
        print(f"[INFO] Running ETS for: {country_dd.value}")

    with plot_out:
        clear_output(wait=True)
    with table_out:
        clear_output(wait=True)

    # Slice country
    sub = df[df["country"] == country_dd.value].copy()
    sub = sub.dropna(subset=[energy_col, "year"]).sort_values("year")

    if sub.empty:
        with metrics_out:
            print(f"No data available for {country_dd.value}")
        return

    # Coerce to numeric y; drop NaNs created by coercion
    y = pd.to_numeric(sub[energy_col], errors="coerce")
    mask = ~y.isna()
    y = y[mask]
    years = sub.loc[mask, "year"].astype(int).values

    if len(y) < 6:
        with metrics_out:
            print(f"Not enough points to fit ETS (have {len(y)}, need ≥ 6).")
        return

    # Shift up if non-positive values exist (for stability; works even without Box-Cox)
    ymin = float(y.min())
    y_shift = 1 - ymin if ymin <= 0 else 0.0
    y_pos = y + y_shift

    # ---- Fit ETS (version-robust) ----
    try:
        fit = fit_ets_robust(y_pos)
    except Exception as e:
        with metrics_out:
            print(f"[ERROR] ETS fit failed: {e}")
        return

    # In-sample fitted (back-transform)
    try:
        fitted_vals = pd.Series(fit.fittedvalues, index=y.index) - y_shift
    except Exception:
        fitted_vals = pd.Series(fit.predict(start=0, end=len(y)-1), index=y.index) - y_shift

    # Metrics on in-sample fit
    rmse_val = rmse(y, fitted_vals)
    mape_val = mape_pct(y, fitted_vals)
    mean_val = float(np.nanmean(y))

    # Forecast next 20 years
    horizon = 20
    fc_pos = fit.forecast(horizon)
    fc = (np.asarray(fc_pos, dtype=float) - y_shift)
    last_year = int(years.max())
    fc_years = np.arange(last_year + 1, last_year + 1 + horizon)

    # Approximate 80% & 95% intervals using residual std and z-scores (no SciPy)
    resid = (y - fitted_vals).astype(float)
    resid_std = float(np.nanstd(resid, ddof=1)) if np.isfinite(resid).sum() > 1 else 0.0
    lower95, upper95 = interval_from_resid(fc, resid_std, z=1.96)
    lower80, upper80 = interval_from_resid(fc, resid_std, z=1.2816)

    # ---- Metrics ----
    with metrics_out:
        print(f"ETS (Exponential Smoothing) — Country: {country_dd.value}")
        print(f"RMSE: {rmse_val:,.3f}")
        print(f"Mean Energy Consumption: {mean_val:,.3f}")
        print("RMSE condition met: RMSE is below 10% of the mean."
              if rmse_val < 0.10 * mean_val else
              "Warning: RMSE condition NOT met: RMSE is above 10% of the mean.")
        print(f"MAPE: {mape_val:,.2f}%")
        print("MAPE condition met: MAPE is below 15%."
              if mape_val < 15.0 else
              "Warning: MAPE condition NOT met: MAPE is above 15%.")

    # ---- Plotly chart ----
    with plot_out:
        fig = go.Figure()
        # Actual
        fig.add_trace(go.Scatter(x=years, y=y, mode="lines+markers", name="Actual"))
        # Forecast
        fig.add_trace(go.Scatter(x=fc_years, y=fc, mode="lines", name="Forecast", line=dict(dash="dash")))
        # 95% band
        fig.add_trace(go.Scatter(
            x=np.concatenate([fc_years, fc_years[::-1]]),
            y=np.concatenate([upper95, lower95[::-1]]),
            fill="toself", name="95% Interval", opacity=0.2, line=dict(width=0)
        ))
        # 80% band
        fig.add_trace(go.Scatter(
            x=np.concatenate([fc_years, fc_years[::-1]]),
            y=np.concatenate([upper80, lower80[::-1]]),
            fill="toself", name="80% Interval", opacity=0.15, line=dict(width=0)
        ))
        fig.update_layout(
            title=f"Exponential Smoothing Forecast for {country_dd.value}",
            xaxis_title="Year",
            yaxis_title=energy_col,
            template="plotly_white",
            legend_title="Series"
        )
        fig.show()

    # ---- Forecast table ----
    with table_out:
        out_df = pd.DataFrame({
            "Year": fc_years,
            "Forecast": np.round(fc, 6),
            "Lower80": np.round(lower80, 6),
            "Upper80": np.round(upper80, 6),
            "Lower95": np.round(lower95, 6),
            "Upper95": np.round(upper95, 6),
        })
        display(out_df)

run_btn.on_click(run_forecast)

display(VBox([
    title_html,
    HBox([country_dd, run_btn]),
    metrics_out,
    plot_out,
    HTML("<h4>Forecast Table</h4>"),
    table_out
]))


[INFO] Using column: primary_energy_consumption for energy consumption
